# State Space Models: Your First Implementation

**Goal:** Build and simulate a state space model in < 2 minutes.

**Time:** 15 minutes

**What you'll build:** A working local level model (random walk + noise) that you can immediately adapt to your own data.

## Quick Win: Simulate a Local Level Model

The local level model is the "hello world" of state space models:
- Hidden state: random walk (true signal)
- Observation: state + noise (what you measure)

Perfect for modeling stock prices, GDP, temperature, etc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# Model: y(t) = α(t) + ε(t),  α(t) = α(t-1) + η(t)
T = 100  # Time periods
sigma_state = 0.5    # State innovation std
sigma_obs = 1.0      # Observation noise std

# Simulate
state = np.zeros(T)
obs = np.zeros(T)

for t in range(T):
    # State evolution
    if t > 0:
        state[t] = state[t-1] + np.random.normal(0, sigma_state)
    else:
        state[t] = 0.0
    
    # Observation
    obs[t] = state[t] + np.random.normal(0, sigma_obs)

# Plot
plt.figure(figsize=(12, 4))
plt.plot(state, label='True State (hidden)', linewidth=2)
plt.plot(obs, 'o', alpha=0.5, label='Observations (noisy)', markersize=4)
plt.legend()
plt.title('Local Level Model: Hidden State vs Noisy Observations')
plt.xlabel('Time')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print(f"State std: {state.std():.2f}")
print(f"Observation std: {obs.std():.2f}")
print(f"Signal-to-noise ratio: {(sigma_state/sigma_obs):.2f}")

**See what happened?** The observations (dots) jump around noisily, but the true state (line) evolves smoothly. 

The Kalman filter will extract that smooth line from just the noisy dots!

## How It Works: State Space Representation

Every state space model has TWO equations:

1. **State equation** (how the hidden state evolves):
   ```
   α(t) = T·α(t-1) + R·η(t)
   ```

2. **Observation equation** (how we measure it):
   ```
   y(t) = Z·α(t) + ε(t)
   ```

For our local level model:
- T = 1 (random walk)
- Z = 1 (observe state directly)
- R = 1 (state noise multiplier)
- Q = σ²_η (state variance)
- H = σ²_ε (observation variance)

In [ ]:
class StateSpaceModel:
    """Simple state space model."""
    
    def __init__(self, T, Z, R, Q, H, a0=None, P0=None):
        """
        T: (m,m) state transition matrix
        Z: (p,m) observation matrix  
        R: (m,r) state noise selector
        Q: (r,r) state noise covariance
        H: (p,p) observation noise covariance
        a0: (m,) initial state mean
        P0: (m,m) initial state covariance
        """
        self.T = np.atleast_2d(T)
        self.Z = np.atleast_2d(Z)
        self.R = np.atleast_2d(R)
        self.Q = np.atleast_2d(Q)
        self.H = np.atleast_2d(H)
        
        self.m = self.T.shape[0]  # state dimension
        self.p = self.Z.shape[0]  # obs dimension
        
        self.a0 = a0 if a0 is not None else np.zeros(self.m)
        self.P0 = P0 if P0 is not None else np.eye(self.m)
        
    def simulate(self, n_periods):
        """Simulate states and observations."""
        states = np.zeros((n_periods, self.m))
        obs = np.zeros((n_periods, self.p))
        
        # Initial state
        states[0] = np.random.multivariate_normal(self.a0, self.P0)
        
        for t in range(n_periods):
            # State evolution
            if t > 0:
                eta = np.random.multivariate_normal(
                    np.zeros(self.Q.shape[0]), self.Q
                )
                states[t] = self.T @ states[t-1] + self.R @ eta
            
            # Observation
            eps = np.random.multivariate_normal(
                np.zeros(self.p), self.H
            )
            obs[t] = self.Z @ states[t] + eps
        
        return obs, states

# Create model
model = StateSpaceModel(
    T=np.array([[1.0]]),
    Z=np.array([[1.0]]),
    R=np.array([[1.0]]),
    Q=np.array([[0.25]]),  # sigma_state^2
    H=np.array([[1.0]])     # sigma_obs^2
)

# Simulate
y, alpha = model.simulate(100)

print(f"Shape of observations: {y.shape}")
print(f"Shape of states: {alpha.shape}")
print("\nReady to filter!")

## Modify This: Experiment with Parameters

Change these values and see what happens:

In [ ]:
# MODIFY THESE VALUES
sigma_state = 0.1    # Try: 0.1, 0.5, 2.0 (how fast state changes)
sigma_obs = 2.0      # Try: 0.5, 1.0, 5.0 (how noisy observations are)
n_periods = 200      # Try: 50, 100, 500

# Create and simulate
model = StateSpaceModel(
    T=np.array([[1.0]]),
    Z=np.array([[1.0]]),
    R=np.array([[1.0]]),
    Q=np.array([[sigma_state**2]]),
    H=np.array([[sigma_obs**2]])
)

y, alpha = model.simulate(n_periods)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Time series
axes[0].plot(alpha, label='True State', linewidth=2)
axes[0].plot(y, 'o', alpha=0.4, label='Observations', markersize=3)
axes[0].legend()
axes[0].set_title(f'σ_state={sigma_state}, σ_obs={sigma_obs}')
axes[0].set_xlabel('Time')
axes[0].grid(True, alpha=0.3)

# Noise characteristics
innovation = y.flatten() - alpha.flatten()
axes[1].hist(innovation, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of Observation Noise')
axes[1].set_xlabel('Error')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Diagnostics
snr = sigma_state / sigma_obs
print(f"Signal-to-Noise Ratio: {snr:.2f}")
if snr < 0.3:
    print("→ Very noisy! Filtering will be crucial.")
elif snr > 2.0:
    print("→ Clean signal! Filtering will be easy.")
else:
    print("→ Moderate noise. Good test case.")

**What did you notice?**
- Small σ_state, large σ_obs → smooth state, very noisy observations
- Large σ_state, small σ_obs → jumpy state, clean observations
- The ratio determines how hard filtering is!

## Real Example: Local Linear Trend

Now let's model something with both level AND slope (like GDP with growth).

In [ ]:
# Model: 
# y(t) = μ(t) + ε(t)
# μ(t) = μ(t-1) + β(t-1) + η_μ(t)
# β(t) = β(t-1) + η_β(t)

# State is [μ, β]'
T_trend = np.array([
    [1.0, 1.0],  # μ(t) = μ(t-1) + β(t-1) + shock
    [0.0, 1.0]   # β(t) = β(t-1) + shock
])

Z_trend = np.array([[1.0, 0.0]])  # Observe level only

R_trend = np.eye(2)

Q_trend = np.diag([0.1, 0.01])  # Level and slope variances

H_trend = np.array([[0.5]])

# Create and simulate
trend_model = StateSpaceModel(T_trend, Z_trend, R_trend, Q_trend, H_trend)
y_trend, states_trend = trend_model.simulate(200)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 9))

# Level
axes[0].plot(states_trend[:, 0], label='True Level (μ)', linewidth=2)
axes[0].plot(y_trend, 'o', alpha=0.3, markersize=2, label='Observations')
axes[0].legend()
axes[0].set_title('Local Linear Trend Model')
axes[0].grid(True, alpha=0.3)

# Slope
axes[1].plot(states_trend[:, 1], label='True Slope (β)', linewidth=2, color='orange')
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].legend()
axes[1].set_title('Growth Rate (Slope)')
axes[1].grid(True, alpha=0.3)

# Observations vs Level
axes[2].plot(y_trend - states_trend[:, 0], label='Observation Noise', alpha=0.7)
axes[2].axhline(0, color='red', linestyle='--')
axes[2].legend()
axes[2].set_title('Measurement Error')
axes[2].set_xlabel('Time')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial slope: {states_trend[0, 1]:.3f}")
print(f"Final slope: {states_trend[-1, 1]:.3f}")
print(f"Average slope: {states_trend[:, 1].mean():.3f}")

**Key insight:** We observe only the level (y), but the model tracks both level AND slope in the hidden state. The Kalman filter will estimate both!

## Copy-Paste Template: AR(1) Model

Production-ready code for an AR(1) process: y(t) = φ·y(t-1) + ε(t)

In [ ]:
def create_ar1_model(phi, sigma_eps, stationary_init=True):
    """
    Create AR(1) model in state space form.
    
    Parameters:
    -----------
    phi : float
        AR coefficient (must be |phi| < 1 for stationarity)
    sigma_eps : float
        Innovation standard deviation
    stationary_init : bool
        Use stationary distribution for initial state
    
    Returns:
    --------
    model : StateSpaceModel
    """
    if abs(phi) >= 1:
        print(f"Warning: |phi|={abs(phi):.2f} >= 1, non-stationary!")
    
    T = np.array([[phi]])
    Z = np.array([[1.0]])
    R = np.array([[1.0]])
    Q = np.array([[sigma_eps**2]])
    H = np.array([[0.0]])  # No observation noise in standard AR
    
    a0 = np.array([0.0])
    P0 = np.array([[sigma_eps**2 / (1 - phi**2)]]) if stationary_init else np.array([[1.0]])
    
    return StateSpaceModel(T, Z, R, Q, H, a0, P0)

# Example: Moderate persistence
ar_model = create_ar1_model(phi=0.7, sigma_eps=1.0)
y_ar, states_ar = ar_model.simulate(200)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(y_ar, linewidth=1.5)
axes[0].set_title('AR(1) Process: φ=0.7')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel('Value')

# ACF
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(y_ar.flatten(), lags=20, ax=axes[1])
axes[1].set_title('Autocorrelation Function')

plt.tight_layout()
plt.show()

print(f"Sample mean: {y_ar.mean():.3f} (should be ≈ 0)")
print(f"Sample std: {y_ar.std():.3f}")
print(f"Theoretical std: {1.0 / np.sqrt(1 - 0.7**2):.3f}")

## Go Deeper (Optional)

**Next steps:**
1. Read [State Space Models guide](../guides/01_state_space_models.md) for theory
2. Implement Kalman filter in [next notebook](02_kalman_filter_visual.ipynb)
3. Apply to real data (see Module 03)

**Try these extensions:**
- Add seasonal component (12-month cycle)
- Implement multivariate observations (p > 1)
- Handle missing data (set some y values to NaN)
- Compare different initialization strategies

**Key takeaways:**
1. State space models separate dynamics (state equation) from measurement (observation equation)
2. This separation makes missing data, irregular sampling, and multivariate systems easy
3. Same framework handles AR, MA, ARMA, structural models, and factor models
4. Simulation is straightforward: iterate the two equations